# Prime Numbers Lab — Notebook 15: Residue-Class Residual Decomposition

**Repo:** `github.com/thinkthoughts/prime-numbers-lab`  
**Notebook purpose:** decompose normalized prime-gap residual structure across residue classes and local arithmetic constraints.

**Core frame:**  
Constraint → structure remains under constraint; drift marks invalid assignments; structure may remain recoverable from partial observation.

Notebook 14 measured residual scaling:

\[
\|\Delta(z,x)\| \sim C(\log x)^{-\alpha}.
\]

Notebook 15 asks where that residual structure lives by decomposing:

\[
\Delta(z;r)=f_{\mathrm{emp}}(z\mid r)-e^{-z}
\]

across residue classes such as \(p_n \bmod 30\) and gap classes such as \((p_{n+1}-p_n) \bmod 30\).


## 0. Setup

This notebook follows the established `prime-numbers-lab` template:

1. define one constraint  
2. generate one dataset  
3. measure what remains under constraint  
4. visualize drift / retention / recoverability  
5. export figures, data, notes, and TeX  
6. package results into a root-level export zip


In [ ]:
# Standard library
from pathlib import Path
import json
import math
import zipfile

# Data / compute
import numpy as np
import pandas as pd

# Plotting
import matplotlib.pyplot as plt

# Notebook identity
NOTEBOOK_ID = "15_residue_class_residual_decomposition"
NOTEBOOK_TITLE = "Residue-Class Residual Decomposition"
REPO_NAME = "prime-numbers-lab"
NOTEBOOK_NUM = NOTEBOOK_ID.split("_")[0]

# Output directories
OUT = Path(NOTEBOOK_ID)
FIG_DIR = OUT / "figures"
DATA_DIR = OUT / "data"
DOCS_DIR = OUT / "docs"
TEX_DIR = OUT / "tex"

for d in [FIG_DIR, DATA_DIR, DOCS_DIR, TEX_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Output directory: {OUT.resolve()}")


## 1. Premise

Notebook 14 found finite-scale residual decay. Notebook 15 asks whether residual drift is uniform or concentrated by arithmetic class.

**Short vocabulary:**

- **Residue class:** a modular bucket such as \(p_n \bmod 30\).
- **Gap class:** a modular bucket such as \((p_{n+1}-p_n) \bmod 30\).
- **Residual mass:** total finite deviation from the exponential baseline.
- **Tail exceedance:** probability that normalized gap \(z\) exceeds a selected threshold.
- **Recoverability:** whether residual drift can be localized to stable arithmetic buckets.

Core question:

> Do prime-gap residuals behave like uniform noise, or do they concentrate in residue-class structure?


## 2. Constraint definition

The normalized prime gap is:

\[
z_n = \frac{p_{n+1}-p_n}{\log p_n}
\]

The exponential reference distribution is:

\[
f_0(z)=e^{-z}
\]

The residue-conditioned residual is:

\[
\Delta(z;r)=f_{\mathrm{emp}}(z\mid r)-f_0(z).
\]


In [ ]:
# Notebook-specific parameters

N_MAX = 2_000_000
RANDOM_SEED = 9423

MODULUS = 30
Z_MAX = 6.0
BIN_COUNT = 90
WINDOW_COUNT = 18
MIN_CLASS_GAPS = 50
MIN_WINDOW_GAPS = 200
TAIL_THRESHOLDS = [1, 2, 3, 4, 5]

rng = np.random.default_rng(RANDOM_SEED)

params = {
    "N_MAX": N_MAX,
    "RANDOM_SEED": RANDOM_SEED,
    "MODULUS": MODULUS,
    "Z_MAX": Z_MAX,
    "BIN_COUNT": BIN_COUNT,
    "WINDOW_COUNT": WINDOW_COUNT,
    "MIN_CLASS_GAPS": MIN_CLASS_GAPS,
    "MIN_WINDOW_GAPS": MIN_WINDOW_GAPS,
    "TAIL_THRESHOLDS": TAIL_THRESHOLDS,
    "NOTEBOOK_ID": NOTEBOOK_ID,
    "NOTEBOOK_TITLE": NOTEBOOK_TITLE,
}

params


## 3. Data generation

Generate primes up to \(N_{\max}\), then compute gaps, normalized gaps, residue classes, and gap classes.


In [ ]:
def generate_primes(n_max: int) -> np.ndarray:
    if n_max < 2:
        return np.array([], dtype=int)

    sieve = np.ones(n_max + 1, dtype=bool)
    sieve[:2] = False

    for i in range(2, int(math.sqrt(n_max)) + 1):
        if sieve[i]:
            sieve[i*i:n_max+1:i] = False

    return np.nonzero(sieve)[0].astype(int)

primes = generate_primes(N_MAX)
gaps = np.diff(primes)
anchors = primes[:-1]
next_primes = primes[1:]
midpoints = np.sqrt(anchors * next_primes)
normalized_gaps = gaps / np.log(anchors)

anchor_mod = anchors % MODULUS
next_mod = next_primes % MODULUS
gap_mod = gaps % MODULUS
transition_label = np.array([f"{a}->{b}" for a, b in zip(anchor_mod, next_mod)])

summary = {
    "n_max": int(N_MAX),
    "prime_count": int(len(primes)),
    "gap_count": int(len(gaps)),
    "first_primes": primes[:10].tolist(),
    "last_primes": primes[-10:].tolist(),
    "mean_gap": float(np.mean(gaps)),
    "mean_normalized_gap": float(np.mean(normalized_gaps)),
    "std_normalized_gap": float(np.std(normalized_gaps)),
    "distinct_anchor_mod_classes": sorted(np.unique(anchor_mod).astype(int).tolist()),
    "distinct_gap_mod_classes": sorted(np.unique(gap_mod).astype(int).tolist()),
}

summary


## 4. Shared measurement functions

Use one baseline and one residual measurement stack across all decompositions.


In [ ]:
def safe_normalize_pdf(pdf: np.ndarray, bin_width: float) -> np.ndarray:
    total = float(np.sum(pdf) * bin_width)
    if total <= 0:
        return pdf
    return pdf / total

def js_divergence_discrete(p_density: np.ndarray, q_density: np.ndarray, bin_width: float) -> float:
    p = np.maximum(p_density * bin_width, 1e-15)
    q = np.maximum(q_density * bin_width, 1e-15)
    p = p / p.sum()
    q = q / q.sum()
    m = 0.5 * (p + q)
    return float(0.5 * (np.sum(p * np.log(p / m)) + np.sum(q * np.log(q / m))))

def empirical_residual_metrics(z_values: np.ndarray, label: str, label_type: str) -> dict:
    hist, _ = np.histogram(z_values, bins=bins, density=True)
    hist = safe_normalize_pdf(hist, bin_width)
    delta = hist - exp_pdf

    row = {
        "label_type": label_type,
        "label": label,
        "count": int(len(z_values)),
        "mean_z": float(np.mean(z_values)),
        "std_z": float(np.std(z_values)),
        "median_z": float(np.median(z_values)),
        "q90_z": float(np.quantile(z_values, 0.90)),
        "q95_z": float(np.quantile(z_values, 0.95)),
        "residual_l1": float(np.sum(np.abs(delta)) * bin_width),
        "residual_l2": float(np.sqrt(np.sum(delta**2) * bin_width)),
        "positive_residual_mass": float(np.sum(np.maximum(delta, 0)) * bin_width),
        "negative_residual_mass": float(np.sum(np.minimum(delta, 0)) * bin_width),
        "js_divergence": js_divergence_discrete(hist, exp_pdf, bin_width),
    }
    for threshold in TAIL_THRESHOLDS:
        row[f"tail_prob_z_gt_{threshold}"] = float(np.mean(z_values > threshold))
    return row

bins = np.linspace(0, Z_MAX, BIN_COUNT + 1)
centers = 0.5 * (bins[:-1] + bins[1:])
bin_width = float(bins[1] - bins[0])
exp_pdf = safe_normalize_pdf(np.exp(-centers), bin_width)

print("Measurement functions ready.")


## 5. Residue-class decomposition

Measure residual mass by:

- anchor residue \(p_n \bmod 30\)
- next-prime residue \(p_{n+1} \bmod 30\)
- gap residue \((p_{n+1}-p_n) \bmod 30\)
- transition residue \((p_n \bmod 30) \to (p_{n+1} \bmod 30)\)


In [ ]:
def class_measurements(values: np.ndarray, z_values: np.ndarray, label_type: str) -> pd.DataFrame:
    rows = []
    for cls in sorted(np.unique(values).tolist(), key=lambda x: str(x)):
        mask = values == cls
        if int(mask.sum()) < MIN_CLASS_GAPS:
            continue
        rows.append(empirical_residual_metrics(z_values[mask], str(cls), label_type))
    return pd.DataFrame(rows)

anchor_mod_df = class_measurements(anchor_mod, normalized_gaps, "anchor_mod30")
next_mod_df = class_measurements(next_mod, normalized_gaps, "next_mod30")
gap_mod_df = class_measurements(gap_mod, normalized_gaps, "gap_mod30")
transition_df = class_measurements(transition_label, normalized_gaps, "transition_mod30")

class_metrics_df = pd.concat([anchor_mod_df, next_mod_df, gap_mod_df, transition_df], ignore_index=True)

measurement = {
    "anchor_mod_class_count": int(len(anchor_mod_df)),
    "next_mod_class_count": int(len(next_mod_df)),
    "gap_mod_class_count": int(len(gap_mod_df)),
    "transition_class_count": int(len(transition_df)),
    "mean_anchor_mod_l1": float(anchor_mod_df["residual_l1"].mean()),
    "mean_gap_mod_l1": float(gap_mod_df["residual_l1"].mean()),
    "mean_transition_l1": float(transition_df["residual_l1"].mean()),
    "max_anchor_mod_l1": float(anchor_mod_df["residual_l1"].max()),
    "max_gap_mod_l1": float(gap_mod_df["residual_l1"].max()),
    "max_transition_l1": float(transition_df["residual_l1"].max()),
}

measurement


## 6. Windowed residue measurements

Build scale windows and measure how class residuals drift with \(x\).


In [ ]:
raw_edges = np.unique(np.logspace(np.log10(100), np.log10(N_MAX), WINDOW_COUNT + 1).astype(int))
raw_edges[0] = 2
raw_edges[-1] = N_MAX

window_rows = []
window_class_rows = []
residual_grid_rows = []

for idx, (left, right) in enumerate(zip(raw_edges[:-1], raw_edges[1:]), start=1):
    window_mask = (anchors >= left) & (anchors < right)
    z_window = normalized_gaps[window_mask]
    if len(z_window) < MIN_WINDOW_GAPS:
        continue
    midpoint = float(math.sqrt(left * right))
    whole = empirical_residual_metrics(z_window, f"window_{idx}", "window")
    whole.update({"window_index": idx, "left": int(left), "right": int(right), "midpoint": midpoint})
    window_rows.append(whole)

    for residue in sorted(np.unique(anchor_mod).astype(int).tolist()):
        class_mask = window_mask & (anchor_mod == residue)
        z_class_window = normalized_gaps[class_mask]
        if len(z_class_window) < MIN_CLASS_GAPS:
            continue
        row = empirical_residual_metrics(z_class_window, str(residue), "anchor_mod30_by_window")
        row.update({"window_index": idx, "left": int(left), "right": int(right), "midpoint": midpoint, "anchor_mod30": int(residue)})
        window_class_rows.append(row)

        hist, _ = np.histogram(z_class_window, bins=bins, density=True)
        hist = safe_normalize_pdf(hist, bin_width)
        delta = hist - exp_pdf
        for c, h, e, d in zip(centers, hist, exp_pdf, delta):
            residual_grid_rows.append({
                "window_index": idx,
                "midpoint": midpoint,
                "anchor_mod30": int(residue),
                "z_center": float(c),
                "empirical_pdf": float(h),
                "exp_pdf": float(e),
                "delta": float(d),
            })

window_metrics_df = pd.DataFrame(window_rows)
window_class_metrics_df = pd.DataFrame(window_class_rows)
residual_grid_df = pd.DataFrame(residual_grid_rows)

window_measurement = {
    "window_count_used": int(len(window_metrics_df)),
    "window_class_rows": int(len(window_class_metrics_df)),
    "mean_window_l1": float(window_metrics_df["residual_l1"].mean()),
    "mean_window_js": float(window_metrics_df["js_divergence"].mean()),
    "mean_window_class_l1": float(window_class_metrics_df["residual_l1"].mean()),
    "mean_window_class_js": float(window_class_metrics_df["js_divergence"].mean()),
}

window_measurement


## 7. CGCS score

The score measures residue-localized residual structure.

Working definition:

\[
CGCS_{\mathrm{residue}}=\frac{1}{1+\overline{\|\Delta_r\|_1}+\overline{JS_r}+\sigma(\|\Delta_r\|_1)}
\]


In [ ]:
anchor_l1_mean = float(anchor_mod_df["residual_l1"].mean())
anchor_js_mean = float(anchor_mod_df["js_divergence"].mean())
anchor_l1_std = float(anchor_mod_df["residual_l1"].std())

cgcs_score = 1.0 / (1.0 + anchor_l1_mean + anchor_js_mean + anchor_l1_std)

cgcs = {
    "score": float(cgcs_score),
    "definition": "1 / (1 + mean residue L1 + mean residue JS + std residue L1)",
    "interpretation": "Closer to 1 indicates smaller and more uniform residue-class residual drift.",
    "mean_anchor_residue_l1": anchor_l1_mean,
    "mean_anchor_residue_js": anchor_js_mean,
    "std_anchor_residue_l1": anchor_l1_std,
}

cgcs


## 8. Visualization

Notebook 15 produces residue-class residual figures:

1. anchor residue residual mass  
2. gap residue residual mass  
3. anchor residue JS divergence  
4. tail exceedance by anchor residue  
5. mean normalized gap by anchor residue  
6. transition residual mass  
7. windowed anchor-residue residual heatmap  
8. strongest residue-class residual curves  
9. windowed JS divergence by scale  
10. residue concentration score  
11. final-window residue PDF comparison


### Figure 1 — anchor residue residual mass


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
plot_df = anchor_mod_df.sort_values("label", key=lambda s: s.astype(int))
ax.bar(plot_df["label"], plot_df["residual_l1"])
ax.set_title("Residual mass by anchor residue mod30")
ax.set_xlabel("p_n mod 30")
ax.set_ylabel("L1 residual mass")
ax.grid(True, axis="y", alpha=0.3)
fig1_path = FIG_DIR / f"{NOTEBOOK_NUM}_anchor_mod30_residual_mass.png"
fig.savefig(fig1_path, dpi=180, bbox_inches="tight")
plt.show()
fig1_path


### Figure 2 — gap residue residual mass


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
plot_df = gap_mod_df.sort_values("label", key=lambda s: s.astype(int))
ax.bar(plot_df["label"], plot_df["residual_l1"])
ax.set_title("Residual mass by gap residue mod30")
ax.set_xlabel("gap mod 30")
ax.set_ylabel("L1 residual mass")
ax.grid(True, axis="y", alpha=0.3)
fig2_path = FIG_DIR / f"{NOTEBOOK_NUM}_gap_mod30_residual_mass.png"
fig.savefig(fig2_path, dpi=180, bbox_inches="tight")
plt.show()
fig2_path


### Figure 3 — anchor residue JS divergence


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
plot_df = anchor_mod_df.sort_values("label", key=lambda s: s.astype(int))
ax.bar(plot_df["label"], plot_df["js_divergence"])
ax.set_title("JS divergence by anchor residue mod30")
ax.set_xlabel("p_n mod 30")
ax.set_ylabel("JS divergence to Exp(1)")
ax.grid(True, axis="y", alpha=0.3)
fig3_path = FIG_DIR / f"{NOTEBOOK_NUM}_anchor_mod30_js_divergence.png"
fig.savefig(fig3_path, dpi=180, bbox_inches="tight")
plt.show()
fig3_path


### Figure 4 — tail exceedance by anchor residue


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
plot_df = anchor_mod_df.sort_values("label", key=lambda s: s.astype(int))
for threshold in [2, 3, 4, 5]:
    ax.plot(plot_df["label"], plot_df[f"tail_prob_z_gt_{threshold}"], marker="o", label=f"P(z>{threshold})")
ax.set_title("Tail exceedance by anchor residue mod30")
ax.set_xlabel("p_n mod 30")
ax.set_ylabel("tail probability")
ax.legend()
ax.grid(True, alpha=0.3)
fig4_path = FIG_DIR / f"{NOTEBOOK_NUM}_tail_exceedance_by_anchor_mod30.png"
fig.savefig(fig4_path, dpi=180, bbox_inches="tight")
plt.show()
fig4_path


### Figure 5 — mean normalized gap by anchor residue


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
plot_df = anchor_mod_df.sort_values("label", key=lambda s: s.astype(int))
ax.plot(plot_df["label"], plot_df["mean_z"], marker="o", label="mean z")
ax.axhline(1.0, linestyle="--", linewidth=1, label="Exp(1) mean")
ax.set_title("Mean normalized gap by anchor residue mod30")
ax.set_xlabel("p_n mod 30")
ax.set_ylabel("mean z")
ax.legend()
ax.grid(True, alpha=0.3)
fig5_path = FIG_DIR / f"{NOTEBOOK_NUM}_mean_z_by_anchor_mod30.png"
fig.savefig(fig5_path, dpi=180, bbox_inches="tight")
plt.show()
fig5_path


### Figure 6 — transition residual mass


In [ ]:
plot_df = transition_df.sort_values("residual_l1", ascending=False).head(20)
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(plot_df["label"], plot_df["residual_l1"])
ax.set_title("Top transition residual masses mod30")
ax.set_xlabel("p_n mod 30 -> p_{n+1} mod 30")
ax.set_ylabel("L1 residual mass")
ax.tick_params(axis="x", rotation=45)
ax.grid(True, axis="y", alpha=0.3)
fig6_path = FIG_DIR / f"{NOTEBOOK_NUM}_transition_mod30_residual_mass_top20.png"
fig.savefig(fig6_path, dpi=180, bbox_inches="tight")
plt.show()
fig6_path


### Figure 7 — windowed anchor-residue residual heatmap


In [ ]:
heat_df = window_class_metrics_df.copy()
heat_df["anchor_mod30"] = heat_df["anchor_mod30"].astype(int)
pivot_l1 = heat_df.pivot_table(index="anchor_mod30", columns="window_index", values="residual_l1", aggfunc="mean").sort_index()
fig, ax = plt.subplots(figsize=(10, 6))
im = ax.imshow(pivot_l1.values, aspect="auto", origin="lower", interpolation="nearest")
ax.set_title("Windowed L1 residual by anchor residue")
ax.set_xlabel("window index")
ax.set_ylabel("p_n mod 30")
ax.set_yticks(np.arange(len(pivot_l1.index)))
ax.set_yticklabels(pivot_l1.index.tolist())
ax.set_xticks(np.arange(len(pivot_l1.columns)))
ax.set_xticklabels(pivot_l1.columns.tolist())
fig.colorbar(im, ax=ax, label="L1 residual")
fig7_path = FIG_DIR / f"{NOTEBOOK_NUM}_windowed_anchor_mod30_l1_heatmap.png"
fig.savefig(fig7_path, dpi=180, bbox_inches="tight")
plt.show()
fig7_path


### Figure 8 — strongest residue-class residual curves


In [ ]:
top_residues = anchor_mod_df.sort_values("residual_l1", ascending=False).head(4)["label"].astype(int).tolist()
fig, ax = plt.subplots(figsize=(9, 5))
for residue in top_residues:
    mask = anchor_mod == residue
    hist, _ = np.histogram(normalized_gaps[mask], bins=bins, density=True)
    hist = safe_normalize_pdf(hist, bin_width)
    delta = hist - exp_pdf
    ax.plot(centers, delta, label=f"mod30={residue}")
ax.axhline(0, linestyle="--", linewidth=1)
ax.set_title("Strongest anchor-residue residual curves")
ax.set_xlabel("normalized gap z")
ax.set_ylabel("Delta(z; residue)")
ax.legend()
ax.grid(True, alpha=0.3)
fig8_path = FIG_DIR / f"{NOTEBOOK_NUM}_strongest_residue_residual_curves.png"
fig.savefig(fig8_path, dpi=180, bbox_inches="tight")
plt.show()
fig8_path


### Figure 9 — windowed JS divergence by scale


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(window_metrics_df["midpoint"], window_metrics_df["js_divergence"], marker="o", label="whole-window JS")
ax.set_xscale("log")
ax.set_title("Windowed JS divergence vs scale")
ax.set_xlabel("window midpoint x")
ax.set_ylabel("JS divergence")
ax.legend()
ax.grid(True, alpha=0.3)
fig9_path = FIG_DIR / f"{NOTEBOOK_NUM}_windowed_js_divergence_vs_scale.png"
fig.savefig(fig9_path, dpi=180, bbox_inches="tight")
plt.show()
fig9_path


### Figure 10 — residue concentration score


In [ ]:
concentration_rows = []
for idx, sub in window_class_metrics_df.groupby("window_index"):
    l1_vals = sub["residual_l1"].values
    concentration_rows.append({"window_index": int(idx), "midpoint": float(sub["midpoint"].iloc[0]), "residue_concentration_score": float(np.max(l1_vals) / (np.mean(l1_vals) + 1e-12)), "mean_l1": float(np.mean(l1_vals)), "max_l1": float(np.max(l1_vals))})
concentration_df = pd.DataFrame(concentration_rows)
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(concentration_df["midpoint"], concentration_df["residue_concentration_score"], marker="o")
ax.set_xscale("log")
ax.set_title("Residue concentration score")
ax.set_xlabel("window midpoint x")
ax.set_ylabel("max residue L1 / mean residue L1")
ax.grid(True, alpha=0.3)
fig10_path = FIG_DIR / f"{NOTEBOOK_NUM}_residue_concentration_score.png"
fig.savefig(fig10_path, dpi=180, bbox_inches="tight")
plt.show()
fig10_path


### Figure 11 — final-window residue PDF comparison


In [ ]:
final_window = int(window_metrics_df["window_index"].iloc[-1])
final_left = int(window_metrics_df["left"].iloc[-1])
final_right = int(window_metrics_df["right"].iloc[-1])
final_mask = (anchors >= final_left) & (anchors < final_right)
final_top = window_class_metrics_df[window_class_metrics_df["window_index"] == final_window].sort_values("residual_l1", ascending=False).head(3)["anchor_mod30"].astype(int).tolist()
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(centers, exp_pdf, linewidth=2, label="Exp(1)")
for residue in final_top:
    mask = final_mask & (anchor_mod == residue)
    if mask.sum() < MIN_CLASS_GAPS:
        continue
    hist, _ = np.histogram(normalized_gaps[mask], bins=bins, density=True)
    hist = safe_normalize_pdf(hist, bin_width)
    ax.plot(centers, hist, marker="o", markersize=3, label=f"final window mod30={residue}")
ax.set_title("Final-window residue PDFs vs Exp(1)")
ax.set_xlabel("normalized gap z")
ax.set_ylabel("density")
ax.legend()
ax.grid(True, alpha=0.3)
fig11_path = FIG_DIR / f"{NOTEBOOK_NUM}_final_window_residue_pdfs_vs_exp1.png"
fig.savefig(fig11_path, dpi=180, bbox_inches="tight")
plt.show()
fig11_path


## 9. Interpretation

Use a short, consistent structure:

1. **What remains under constraint?**  
2. **What drifts?**  
3. **What appears recoverable?**  
4. **What should not be overclaimed?**


In [ ]:
strongest_anchor = anchor_mod_df.sort_values("residual_l1", ascending=False).iloc[0]
strongest_gap = gap_mod_df.sort_values("residual_l1", ascending=False).iloc[0]
strongest_transition = transition_df.sort_values("residual_l1", ascending=False).iloc[0]

interpretation = f"""
# {NOTEBOOK_TITLE}

## Constraint result

This notebook decomposes normalized prime-gap residuals by residue class.

The residue-conditioned residual is:

$$
\\Delta(z;r)=f_{{emp}}(z\\mid r)-e^{{-z}}.
$$

## Remains under constraint

The exponential envelope remains visible globally, while residue-conditioned buckets retain structured finite deviations.

## Drift

The strongest anchor-residue class by L1 residual is:

- anchor residue mod30 = {strongest_anchor['label']}
- L1 residual = {strongest_anchor['residual_l1']:.6f}
- JS divergence = {strongest_anchor['js_divergence']:.6f}

The strongest gap-residue class by L1 residual is:

- gap residue mod30 = {strongest_gap['label']}
- L1 residual = {strongest_gap['residual_l1']:.6f}
- JS divergence = {strongest_gap['js_divergence']:.6f}

The strongest transition by L1 residual is:

- transition = {strongest_transition['label']}
- L1 residual = {strongest_transition['residual_l1']:.6f}
- JS divergence = {strongest_transition['js_divergence']:.6f}

## Recoverability

Residual structure is recoverable as a finite residue-class diagnostic:

- anchor residue residual mass
- gap residue residual mass
- transition residual mass
- windowed residue heatmaps
- strongest residue residual curves

## CGCS score

The residue agreement score is:

$$
CGCS_{{residue}} =
\\frac{{1}}{{1+\\overline{{\\|\\Delta_r\\|_1}}+\\overline{{JS_r}}+\\sigma(\\|\\Delta_r\\|_1)}}.
$$

Measured score:

$$
CGCS_{{residue}} = {cgcs_score:.6f}.
$$

## Caution

This notebook does not prove an asymptotic theorem.

It measures finite residue-class residual structure relative to the exponential heuristic.
""".strip()

print(interpretation)


## 10. Export data, notes, figures index, math, and TeX

This block writes reusable artifacts:

- CSV summaries
- JSON metadata
- Markdown interpretation with embedded figure links
- Markdown design notes
- TeX results snippet
- standalone TeX math notes


In [ ]:
summary_df = pd.DataFrame([{**params, **summary, **measurement, **window_measurement, "cgcs_score": cgcs["score"], "cgcs_definition": cgcs["definition"]}])

summary_path = DATA_DIR / f"{NOTEBOOK_NUM}_summary.csv"
class_metrics_path = DATA_DIR / f"{NOTEBOOK_NUM}_class_metrics.csv"
anchor_mod_path = DATA_DIR / f"{NOTEBOOK_NUM}_anchor_mod30_metrics.csv"
next_mod_path = DATA_DIR / f"{NOTEBOOK_NUM}_next_mod30_metrics.csv"
gap_mod_path = DATA_DIR / f"{NOTEBOOK_NUM}_gap_mod30_metrics.csv"
transition_path = DATA_DIR / f"{NOTEBOOK_NUM}_transition_mod30_metrics.csv"
window_metrics_path = DATA_DIR / f"{NOTEBOOK_NUM}_window_metrics.csv"
window_class_metrics_path = DATA_DIR / f"{NOTEBOOK_NUM}_window_class_metrics.csv"
residual_grid_path = DATA_DIR / f"{NOTEBOOK_NUM}_residual_grid.csv"
concentration_path = DATA_DIR / f"{NOTEBOOK_NUM}_concentration.csv"
metadata_path = DATA_DIR / f"{NOTEBOOK_NUM}_metadata.json"
interpretation_md_path = DOCS_DIR / f"{NOTEBOOK_NUM}_interpretation.md"
design_notes_md_path = DOCS_DIR / f"{NOTEBOOK_NUM}_design_notes.md"
summary_tex_path = TEX_DIR / f"{NOTEBOOK_NUM}_summary_snippet.tex"
math_tex_path = TEX_DIR / f"{NOTEBOOK_NUM}_math_notes.tex"

summary_df.to_csv(summary_path, index=False)
class_metrics_df.to_csv(class_metrics_path, index=False)
anchor_mod_df.to_csv(anchor_mod_path, index=False)
next_mod_df.to_csv(next_mod_path, index=False)
gap_mod_df.to_csv(gap_mod_path, index=False)
transition_df.to_csv(transition_path, index=False)
window_metrics_df.to_csv(window_metrics_path, index=False)
window_class_metrics_df.to_csv(window_class_metrics_path, index=False)
residual_grid_df.to_csv(residual_grid_path, index=False)
concentration_df.to_csv(concentration_path, index=False)

figure_paths = [fig1_path, fig2_path, fig3_path, fig4_path, fig5_path, fig6_path, fig7_path, fig8_path, fig9_path, fig10_path, fig11_path]
metadata = {
    "params": params,
    "summary": summary,
    "measurement": measurement,
    "window_measurement": window_measurement,
    "cgcs": cgcs,
    "strongest_anchor": strongest_anchor.to_dict(),
    "strongest_gap": strongest_gap.to_dict(),
    "strongest_transition": strongest_transition.to_dict(),
    "figures": [str(p) for p in figure_paths],
    "data": {
        "summary": str(summary_path),
        "class_metrics": str(class_metrics_path),
        "anchor_mod30_metrics": str(anchor_mod_path),
        "next_mod30_metrics": str(next_mod_path),
        "gap_mod30_metrics": str(gap_mod_path),
        "transition_mod30_metrics": str(transition_path),
        "window_metrics": str(window_metrics_path),
        "window_class_metrics": str(window_class_metrics_path),
        "residual_grid": str(residual_grid_path),
        "concentration": str(concentration_path),
    },
    "docs": {"interpretation": str(interpretation_md_path), "design_notes": str(design_notes_md_path)},
    "tex": {"summary_snippet": str(summary_tex_path), "math_notes": str(math_tex_path)},
}
metadata_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")

figures_md = "\n\n## Figures\n\n"
for i, fig in enumerate(figure_paths, start=1):
    figures_md += f"### Figure {i} — {fig.stem.replace('_', ' ').title()}\n\n"
    figures_md += f"![Figure {i}](../figures/{fig.name})\n\n"
interpretation_md_path.write_text(interpretation + figures_md, encoding="utf-8")

design_notes = f"""
# Design Notes — {NOTEBOOK_TITLE}

## Notebook role

Notebook 15 follows Notebook 14 by localizing normalized prime-gap residuals to residue classes.

## Constraint

The residue-conditioned residual is:

$$
\\Delta(z;r)=f_{{emp}}(z\\mid r)-e^{{-z}}.
$$

## Measurement

Metrics include residual L1, residual L2, JS divergence, positive/negative residual mass, and tail exceedance by residue class.

## CGCS score

$$
CGCS_{{residue}} =
\\frac{{1}}{{1+\\overline{{\\|\\Delta_r\\|_1}}+\\overline{{JS_r}}+\\sigma(\\|\\Delta_r\\|_1)}}.
$$

## Handoff

Notebook 16 should compare residue-class residual structure against randomized or shuffled controls.
""".strip()
design_notes_md_path.write_text(design_notes + "\n", encoding="utf-8")

summary_tex = rf"""
\section*{{{NOTEBOOK_TITLE}}}

This notebook decomposes normalized prime-gap residuals by residue class.

\[
\Delta(z;r)=f_{{emp}}(z\mid r)-e^{{-z}}
\]

\begin{{itemize}}
  \item Anchor residue classes used: {measurement['anchor_mod_class_count']}
  \item Gap residue classes used: {measurement['gap_mod_class_count']}
  \item Transition classes used: {measurement['transition_class_count']}
  \item Mean anchor-residue $L_1$: {measurement['mean_anchor_mod_l1']:.6f}
  \item Mean gap-residue $L_1$: {measurement['mean_gap_mod_l1']:.6f}
  \item Mean transition $L_1$: {measurement['mean_transition_l1']:.6f}
  \item CGCS residue score: {cgcs_score:.6f}
\end{{itemize}}

Residue-class decomposition is treated as a finite-scale empirical diagnostic, not an asymptotic proof.
""".strip()
summary_tex_path.write_text(summary_tex + "\n", encoding="utf-8")

math_tex = rf"""
\documentclass{{article}}
\usepackage{{amsmath}}
\usepackage{{amssymb}}
\usepackage[margin=1in]{{geometry}}

\begin{{document}}

\section*{{Math Notes: {NOTEBOOK_TITLE}}}

\subsection*{{Normalized gap}}

\[
z_n = \frac{{p_{{n+1}}-p_n}}{{\log p_n}}
\]

\subsection*{{Residue classes}}

\[
r_n = p_n \bmod 30
\]

\[
g_n = (p_{{n+1}}-p_n) \bmod 30
\]

\subsection*{{Exponential baseline}}

\[
f_0(z)=e^{{-z}}, \quad z\ge 0
\]

\subsection*{{Residue-conditioned residual}}

\[
\Delta(z;r)=f_{{emp}}(z\mid r)-f_0(z)
\]

\subsection*{{Residue CGCS score}}

\[
CGCS_{{residue}} =
\frac{{1}}{{1+\overline{{\|\Delta_r\|_1}}+\overline{{JS_r}}+\sigma(\|\Delta_r\|_1)}}
\]

\subsection*{{Interpretation}}

If residual mass concentrates by residue class, finite prime-gap drift is not uniform across modular buckets.

\end{{document}}
""".strip()
math_tex_path.write_text(math_tex + "\n", encoding="utf-8")

summary_path, class_metrics_path, anchor_mod_path, gap_mod_path, transition_path, window_metrics_path, window_class_metrics_path, residual_grid_path, concentration_path, metadata_path, interpretation_md_path, design_notes_md_path, summary_tex_path, math_tex_path


## 11. Optional results bundle

This creates a root-level export zip containing figures, data, docs, and TeX outputs.


In [ ]:
EXPORT_NAME = f"{NOTEBOOK_ID}_export.zip"

with zipfile.ZipFile(EXPORT_NAME, "w", zipfile.ZIP_DEFLATED) as z:
    for folder in [DOCS_DIR, DATA_DIR, FIG_DIR, TEX_DIR]:
        for path in folder.rglob("*"):
            if path.is_file():
                z.write(path, path.as_posix())

print(f"Export ready: {EXPORT_NAME}")
print("Tip: uncomment Colab lines below to download.")

# --- Optional Colab download ---
# Uncomment the lines below when running in Colab
#
# from google.colab import files
# files.download(EXPORT_NAME)


## 12. Next notebook handoff

Next:

> Notebook 16 should compare residue-class residual structure against randomized or shuffled controls.


In [ ]:
next_step = "Notebook 16: randomized residue-control validation."
print(next_step)
